In [ ]:
# 1) Colab runtime & GPU selection checks
import sys
import os

print('Python version:', sys.version)
# GPU check
try:
    gpu_info = os.popen('nvidia-smi -L').read()
    if gpu_info:
        print('GPU detected:')
        print(gpu_info)
    else:
        print('No GPU detected via nvidia-smi (this is ok for CPU runs).')
except Exception as e:
    print('nvidia-smi check failed:', e)

# Torch device check
try:
    import torch
    print('torch version:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('Device count:', torch.cuda.device_count())
except Exception as e:
    print('torch import failed:', e)

---

2) Mount Google Drive

Use the cell below to mount Google Drive and set workspace paths if you want outputs persisted in Drive.

---

In [ ]:
# Mount Google Drive (optional)
try:
    from google.colab import drive
    DRIVE_AVAILABLE = True
except Exception:
    DRIVE_AVAILABLE = False

if DRIVE_AVAILABLE:
    print('Google Colab Drive utilities available. Use drive.mount to persist outputs.')
else:
    print('Not running in Colab or google.colab not available; skip Drive mounting.')

# If running in Colab and you want to persist outputs, uncomment and run:
# from google.colab import drive
# drive.mount('/content/drive')
# then set WORKSPACE = '/content/drive/MyDrive/delta_meth' or similar

---

3) Install dependencies

This cell installs Python packages required to run the pipeline. It will attempt to install from the cloned repository `requirements.txt` when available; otherwise it falls back to direct pip installs for the core libraries.

---

In [ ]:
# Install dependencies
import os

# If you cloned the repo to /content/delta_meth and it has requirements.txt, install from it.
req_path = '/content/delta_meth/requirements.txt'
if os.path.exists(req_path):
    print('Installing from repo requirements.txt...')
    os.system(f'pip install -q -r {req_path}')
else:
    print('requirements.txt not found in /content/delta_meth, installing core packages...')
    os.system('pip install -q sentence-transformers transformers torch scikit-learn pyyaml')

# Verify imports
try:
    import sentence_transformers
    import transformers
    import torch
    import sklearn
    import yaml
    print('Imports OK: ', sentence_transformers.__version__, transformers.__version__, torch.__version__)
except Exception as e:
    print('Import check failed:', e)
    # Continue; we'll provide a mock fallback later


---

4) Clone repository & checkout branch

Provide a `GITHUB_URL` variable to clone the repository into `/content/delta_meth`. Uncomment and set your GitHub URL as needed.

---

In [ ]:
# Clone repository (Option A: clone from GitHub)
GITHUB_URL = ''  # e.g. 'https://github.com/username/repo.git'
if GITHUB_URL:
    print('Cloning from', GITHUB_URL)
    os.system('rm -rf /content/delta_meth')
    os.system(f'git clone {GITHUB_URL} /content/delta_meth')
else:
    print('GITHUB_URL not set. If you uploaded the repo zip, unzip it to /content/delta_meth manually.')

# Option B: upload a zip archive via the Colab file uploader (manual step in UI)

---

5) Set environment variables / secrets

Load secrets from Drive or prompt the user. Keep secrets out of notebook outputs.

---

In [ ]:
# Secrets / environment variables (optional)
import os
from getpass import getpass

# Example: set a GH token for cloning private repo
# GH_TOKEN = getpass('GitHub token (leave blank for public repos): ')
# if GH_TOKEN:
#     os.environ['GITHUB_TOKEN'] = GH_TOKEN

print('You can set secrets via Drive or interactive prompts. Do not store secrets in the notebook outputs.')

---

6) Prepare dataset / sync data

Copy relevant `data/raw` files into the local workspace if needed. If you cloned the repo, the data directory should already be present.

---

In [ ]:
# Check for data/raw files
import os
p = '/content/delta_meth/data/raw'
if os.path.exists(p):
    print('Found data/raw with files:')
    print(os.listdir(p))
else:
    print('No data/raw directory found at /content/delta_meth/data/raw. If you uploaded the repo, ensure data/raw is present.')


---

7) Write pipeline configuration (YAML/JSON)

Load and print the active configuration from `configs/config.yaml` in the cloned repo. If the file is not present, show a default config and write it to disk.

---

In [ ]:
# Load config
import yaml
from pathlib import Path

cfg_path = Path('/content/delta_meth/configs/config.yaml')
if cfg_path.exists():
    cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
    print('Loaded config:')
    print(cfg)
else:
    print('Config not found at /content/delta_meth/configs/config.yaml. Showing default config:')
    cfg = {
        'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
        'nli_model': 'joeddav/xlm-roberta-large-xnli',
        'chunk_size': 150,
        'similarity_threshold': 0.5,
        'nli_threshold': 0.7,
    }
    print(cfg)
    # Optionally write it out
    # cfg_path.parent.mkdir(parents=True, exist_ok=True)
    # cfg_path.write_text(yaml.safe_dump(cfg, allow_unicode=True), encoding='utf-8')


---

8) Implement pipeline runner and mock fallback

This cell imports the pipeline functions and attempts to run them. If heavy model imports fail, the notebook falls back to a mock runner that simulates embeddings and NLI labels for demonstration.

---

In [ ]:
# Run pipeline with fallback
import json
from pathlib import Path

# Ensure repo path is on sys.path
import sys
repo_path = '/content/delta_meth'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

mock_mode = False

try:
    # Try importing the real pipeline
    from src.pipeline.run_pipeline import run_pipeline
    from src.alignment.embeddings import encode_chunks
    from src.nli.nli_model import predict_nli
    print('Pipeline modules imported successfully; running real pipeline...')
    contradiction, detailed = run_pipeline(config_path='/content/delta_meth/configs/config.yaml', verbose=True)
except Exception as e:
    print('Real pipeline failed or models not available; switching to mock mode. Error:', e)
    mock_mode = True

if mock_mode:
    # Minimal mock implementation using local functions to simulate outputs
    from src.preprocessing.chunking import chunk_notes
    a = Path('/content/delta_meth/data/raw/dummy_note_20260212.txt')
    b = Path('/content/delta_meth/data/raw/dummy_note_20260213.txt')
    note_a = a.read_text(encoding='utf-8') if a.exists() else 'Patient no fever.'
    note_b = b.read_text(encoding='utf-8') if b.exists() else 'Patient has fever.'
    chunks_a = chunk_notes(note_a, chunk_size=50)
    chunks_b = chunk_notes(note_b, chunk_size=50)
    # Simulate alignment: pair each chunk i with j=i
    detailed = []
    for i, ca in enumerate(chunks_a):
        if i < len(chunks_b):
            cb = chunks_b[i]
        else:
            cb = chunks_b[-1] if chunks_b else ''
        # Mock sim score and NLI: if Greek words differ contain 'πυρετό' vs 'δεν' set contradiction
        sim_score = 0.6
        if 'πυρετό' in ca or 'πυρετό' in cb:
            label = 'contradiction'
            conf = 0.95
        else:
            label = 'neutral'
            conf = 0.6
        entry = {
            'i': i,
            'j': i if i < len(chunks_b) else len(chunks_b)-1,
            'sim_score': sim_score,
            'nli_label': label,
            'nli_confidence': conf,
            'chunk1': ca,
            'chunk2': cb,
        }
        detailed.append(entry)
    # Select highest-confidence contradiction
    contradictions = [d for d in detailed if d['nli_label']=='contradiction']
    contradiction = max(contradictions, key=lambda x: x['nli_confidence']) if contradictions else None

# Save results to results/run_output.json
out = {'contradiction': contradiction, 'detailed_pairs': detailed}
out_path = Path('/content/delta_meth/results')
out_path.mkdir(parents=True, exist_ok=True)
(out_path / 'run_output.json').write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding='utf-8')

print('\nSaved run_output.json to /content/delta_meth/results/run_output.json')
print('Contradiction:', contradiction)


---

9) Run unit tests and validation (optional)

You can run `pytest` on specific test files if present in the repository. This next cell shows how to run tests and capture results.

---

In [ ]:
# Optional: run pytest if available
import subprocess
try:
    res = subprocess.run(['pytest', '-q'], capture_output=True, text=True, cwd=repo_path)
    print('pytest exit code:', res.returncode)
    print(res.stdout)
    print(res.stderr)
except FileNotFoundError:
    print('pytest not installed. You can install with pip install pytest')
except Exception as e:
    print('pytest run failed:', e)

---

11) Save artifacts and logs to Drive / cloud

This cell demonstrates copying the `results` folder to Drive or cloud storage for persistence.

---

In [ ]:
# Copy results to Drive (if mounted)
from pathlib import Path
import shutil

drive_results = Path('/content/drive/MyDrive/delta_meth_results')
local_results = Path('/content/delta_meth/results')
if drive_results.exists():
    print('Copying results to Drive...')
    if local_results.exists():
        shutil.copytree(local_results, drive_results, dirs_exist_ok=True)
        print('Copied to', drive_results)
    else:
        print('No local results to copy')
else:
    print('Drive results path not found; mount Drive to use this feature.')

---

12) Export notebook to script & run in background

Convert the notebook to a script and optionally run it in the background (for long runs). Example uses `nbconvert` to export.

---

In [ ]:
# Export notebook to script
try:
    os.system('jupyter nbconvert --to script notebooks/run_pipeline_colab.ipynb')
    print('Exported to notebooks/run_pipeline_colab.py')
except Exception as e:
    print('Export failed:', e)

# Example to run background (beware in Colab; use for long local runs):
# nohup python notebooks/run_pipeline_colab.py > run.log 2>&1 &


# Run DiaShift pipeline (Colab)

This notebook installs dependencies, loads the repository, and runs the DiaShift pipeline (chunk -> embeddings -> alignment -> NLI filtering) using the configuration in `configs/config.yaml`.

Usage:
- Set `GITHUB_URL` and clone the repo, or upload the repository archive.
- Choose Runtime > Change runtime type > GPU for faster model downloads.

The notebook will write `results/run_output.json` with the detailed aligned pairs and NLI labels.